# 05 — Final Load Prep
### NST DVA Capstone 2 | NYC Airbnb Smart Booking Intelligence

This notebook prepares the final dataset for downstream use in Tableau and statistical reporting.

| Step | Description |
|------|-------------|
| **1** | Feature Engineering — `last_review_month`, `last_review_day_of_week`, `price_bucket`, `occupancy_rate`, `demand_score`, `value_score`, `availability_category`, `price_category` |
| **2** | Column Renaming — standardise all column names to `snake_case` |
| **3** | Drop helper columns — remove `is_price_outlier` and intermediate fields |
| **4** | Final Validation — shape, dtypes, null counts, value ranges |
| **5** | Save final dataset — `data/processed/airbnb_nyc_final.csv` |

---

## 0. Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings("ignore")

# ── Paths ──────────────────────────────────────────────────────────────────────
INPUT_PATH  = "data/processed/processed_AB_NYC_2019.csv"
OUTPUT_PATH = "data/processed/airbnb_nyc_final.csv"

os.makedirs("data/processed", exist_ok=True)
print("✅ Libraries loaded")
print(f"   Input  → {INPUT_PATH}")
print(f"   Output → {OUTPUT_PATH}")

## 1. Load Dataset

In [ ]:
df = pd.read_csv(INPUT_PATH)

print(f"Shape  : {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Columns: {list(df.columns)}")
df.head()

In [ ]:
# Quick overview of data types and null counts before transformation
print("─" * 55)
print("Null counts before feature engineering:")
print("─" * 55)
print(df.isnull().sum().to_string())

---
## 2. Feature Engineering

Eight new columns are added based on existing fields. These columns power the KPIs and filters used in all four Tableau dashboards.

### 2a. Last Review Month & Day of Week

In [ ]:
# Parse date column
df["last_review"] = pd.to_datetime(df["last_review"], errors="coerce")

# Extract month name and day of week
df["last_review_month"]       = df["last_review"].dt.month_name()
df["last_review_day_of_week"] = df["last_review"].dt.day_name()

# Fill nulls for listings with no reviews
df["last_review_month"]       = df["last_review_month"].fillna("No Reviews")
df["last_review_day_of_week"] = df["last_review_day_of_week"].fillna("No Reviews")

print("last_review_month — value counts:")
print(df["last_review_month"].value_counts())

### 2b. Price Bucket

In [ ]:
def assign_price_bucket(price):
    if price == 0:
        return "Unknown"
    elif price <= 75:
        return "Budget (≤$75)"
    elif price <= 175:
        return "Mid-Range ($76–$175)"
    else:
        return "Luxury (>$175)"

df["price_bucket"] = df["price"].apply(assign_price_bucket)

print("price_bucket — value counts:")
print(df["price_bucket"].value_counts())

### 2c. Occupancy Rate

In [ ]:
# Formula: (365 - availability_365) / 365
# 0.0 = fully available (never booked), 1.0 = fully booked
df["occupancy_rate"] = ((365 - df["availability_365"]) / 365).round(4)
df["occupancy_rate"] = df["occupancy_rate"].clip(0, 1)

print(f"occupancy_rate → min: {df['occupancy_rate'].min():.4f} | "
      f"max: {df['occupancy_rate'].max():.4f} | "
      f"mean: {df['occupancy_rate'].mean():.4f}")

### 2d. Demand Score

In [ ]:
# Fill reviews_per_month nulls with 0 (listings with no reviews)
df["reviews_per_month"] = df["reviews_per_month"].fillna(0)

# Formula: reviews_per_month × (1 - availability_365 / 365)
# Higher score = frequently reviewed + low availability (high demand)
df["demand_score"] = (
    df["reviews_per_month"] * (1 - df["availability_365"] / 365)
).round(4)

print(f"demand_score → min: {df['demand_score'].min():.4f} | "
      f"max: {df['demand_score'].max():.4f} | "
      f"mean: {df['demand_score'].mean():.4f}")

### 2e. Value Score

In [ ]:
# Formula: reviews_per_month / price_per_person  → normalised to 0–1
# Higher score = more reviews per dollar = better value for money
df["value_score_raw"] = np.where(
    df["price_per_person"] > 0,
    df["reviews_per_month"] / df["price_per_person"],
    0
)
v_min = df["value_score_raw"].min()
v_max = df["value_score_raw"].max()
df["value_score"] = ((df["value_score_raw"] - v_min) / (v_max - v_min)).round(4)
df.drop(columns=["value_score_raw"], inplace=True)

print(f"value_score → min: {df['value_score'].min():.4f} | "
      f"max: {df['value_score'].max():.4f} | "
      f"mean: {df['value_score'].mean():.4f}")

### 2f. Availability Category

In [ ]:
def assign_availability_category(days):
    if days <= 90:
        return "Low (≤90 days)"
    elif days <= 270:
        return "Medium (91–270 days)"
    else:
        return "High (>270 days)"

df["availability_category"] = df["availability_365"].apply(assign_availability_category)

print("availability_category — value counts:")
print(df["availability_category"].value_counts())

### 2g. Price Category

In [ ]:
def assign_price_category(price):
    if price == 0:
        return "Unknown"
    elif price <= 75:
        return "Budget"
    elif price <= 175:
        return "Mid-Range"
    else:
        return "Luxury"

df["price_category"] = df["price"].apply(assign_price_category)

print("price_category — value counts:")
print(df["price_category"].value_counts())

In [ ]:
# Preview all new engineered columns
new_cols = [
    "last_review_month", "last_review_day_of_week",
    "price_bucket", "occupancy_rate", "demand_score",
    "value_score", "availability_category", "price_category"
]
df[new_cols].head(10)

---
## 3. Column Renaming to snake_case

All column names are standardised for clarity and consistency across notebooks, scripts, and Tableau.

In [ ]:
rename_map = {
    "id"                             : "listing_id",
    "name"                           : "listing_name",
    "host_id"                        : "host_id",
    "host_name"                      : "host_name",
    "neighbourhood_group"            : "borough",
    "neighbourhood"                  : "neighbourhood",
    "latitude"                       : "latitude",
    "longitude"                      : "longitude",
    "room_type"                      : "room_type",
    "price"                          : "price_usd",
    "minimum_nights"                 : "minimum_nights",
    "number_of_reviews"              : "total_reviews",
    "last_review"                    : "last_review_date",
    "reviews_per_month"              : "reviews_per_month",
    "calculated_host_listings_count" : "host_total_listings",
    "availability_365"               : "availability_days",
    "price_per_person"               : "price_per_person_usd",
}

df.rename(columns=rename_map, inplace=True)

print("✅ Columns renamed. New column list:")
for i, col in enumerate(df.columns, 1):
    print(f"  {i:02d}. {col}")

---
## 4. Drop Helper Columns

In [ ]:
cols_to_drop = ["is_price_outlier"]   # add any other intermediate columns here
existing_drops = [c for c in cols_to_drop if c in df.columns]

if existing_drops:
    df.drop(columns=existing_drops, inplace=True)
    print(f"✅ Dropped columns: {existing_drops}")
else:
    print("ℹ️  No helper columns found to drop — dataset already clean.")

print(f"\nDataset shape after drop: {df.shape[0]:,} rows × {df.shape[1]} columns")

---
## 5. Final Validation

In [ ]:
print(f"Final shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
df.dtypes

In [ ]:
# Null check
nulls = df.isnull().sum()
null_cols = nulls[nulls > 0]
if not null_cols.empty:
    print("⚠️  Columns with nulls:")
    print(null_cols.to_string())
else:
    print("✅ No unexpected nulls (last_review_date nulls are expected for no-review listings)")

In [ ]:
# Numeric range validation
numeric_checks = {
    "price_usd"           : (0, 10000),
    "occupancy_rate"      : (0, 1),
    "demand_score"        : (0, None),
    "value_score"         : (0, 1),
    "availability_days"   : (0, 365),
    "minimum_nights"      : (1, None),
    "price_per_person_usd": (0, None),
}

print(f"{'Column':<25} {'Min':>10} {'Max':>10}  {'Status'}")
print("─" * 60)
for col, (lo, hi) in numeric_checks.items():
    if col in df.columns:
        col_min = df[col].min()
        col_max = df[col].max()
        flag = "✅"
        if lo is not None and col_min < lo: flag = "⚠️  below min"
        if hi is not None and col_max > hi: flag = "⚠️  above max"
        print(f"{col:<25} {col_min:>10.4f} {col_max:>10.4f}  {flag}")

In [ ]:
# Data quality flags
zero_price   = (df["price_usd"] == 0).sum()
multi_hosts  = (df["host_total_listings"] > 1).sum()
long_stays   = (df["minimum_nights"] > 365).sum()

print(f"⚠️  Listings with price = $0        : {zero_price}  → exclude from price KPIs")
print(f"ℹ️  Multi-listing host rows          : {multi_hosts:,} ({multi_hosts/len(df)*100:.1f}%)")
print(f"ℹ️  Listings with min_nights > 365  : {long_stays}  → long-stay / anomalous")

In [ ]:
# Final preview
df.head()

---
## 6. Save Final Dataset

In [ ]:
df.to_csv(OUTPUT_PATH, index=False)

print("=" * 55)
print("✅ 05_final_load_prep COMPLETE")
print("=" * 55)
print(f"   Saved to : {OUTPUT_PATH}")
print(f"   Rows     : {df.shape[0]:,}")
print(f"   Columns  : {df.shape[1]}")
print()
print("   Next step → Connect Tableau to airbnb_nyc_final.csv")